# 01 – Exploratory Data Analysis (EDA)

This notebook provides an exploratory look at the Monte Carlo samples used in the feasibility study of transverse tau polarization in semitauonic B decays at Belle II.

**Goals:**
- Load signal and background MC samples
- Inspect kinematic distributions of key variables
- Identify variables with good signal/background separation power
- Study correlations between variables

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot

# Style
try:
    import mplhep as hep
    hep.style.use(hep.style.Belle2)
except ImportError:
    pass

plt.rcParams['figure.dpi'] = 120

## 1. Load Data

Update the paths below to point to the ntuple files on Athena.

In [ ]:
# ─── Paths – update as needed ────────────────────────────────────────────────
SIGNAL_FILE = "/path/to/signal_mc.root"
BKG_FILE    = "/path/to/background_mc.root"
TREE_NAME   = "ntuple"          # ROOT tree name
# ─────────────────────────────────────────────────────────────────────────────

with uproot.open(SIGNAL_FILE) as f:
    sig = f[TREE_NAME].arrays(library="pd")

with uproot.open(BKG_FILE) as f:
    bkg = f[TREE_NAME].arrays(library="pd")

print(f"Signal events : {len(sig):,}")
print(f"Background events: {len(bkg):,}")
sig.head()

## 2. Key Variable Distributions

In [ ]:
# List of variables to inspect – update to match your ntuple branches
VARIABLES = [
    "M2_miss",    # missing mass squared
    "q2",         # momentum transfer squared
    "E_miss",     # missing energy
    "p_D_cms",    # D meson momentum in CMS
    "cos_theta_tau",  # cosine of tau helicity angle
]

fig, axes = plt.subplots(1, len(VARIABLES), figsize=(5 * len(VARIABLES), 4))

for ax, var in zip(axes, VARIABLES):
    if var not in sig.columns:
        ax.set_title(f"{var}\n(not found)")
        continue
    bins = np.linspace(sig[var].quantile(0.01), sig[var].quantile(0.99), 50)
    ax.hist(sig[var], bins=bins, density=True, alpha=0.6, label="Signal", color="steelblue")
    ax.hist(bkg[var], bins=bins, density=True, alpha=0.6, label="Background", color="tomato")
    ax.set_xlabel(var)
    ax.set_ylabel("Normalized counts")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../docs/eda_distributions.pdf")
plt.show()

## 3. Correlation Matrix (Signal)

In [ ]:
available = [v for v in VARIABLES if v in sig.columns]

corr = sig[available].corr()

fig, ax = plt.subplots(figsize=(len(available) + 1, len(available)))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(available)), available, rotation=45, ha="right")
ax.set_yticks(range(len(available)), available)
plt.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Signal correlation matrix")
plt.tight_layout()
plt.show()

## 4. Summary Statistics

In [ ]:
print("=== Signal ===")
display(sig[available].describe())
print("=== Background ===")
display(bkg[available].describe())